# 01 - Simple Linear Regression End-to-End

Executed notebook for simple linear regression. It includes data audit, visualization cells, manual OLS, scikit-learn implementation, statsmodels inference, metrics, residual analysis, and exercises.

## 1. Problem statement

Predict `exam_score` from `study_hours`.

```text
exam_score = beta_0 + beta_1 * study_hours + error
```

The slope tells us the expected score change for one additional study hour.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.api as sm

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 140)
DATA_DIR = Path.cwd().parent / 'data' if (Path.cwd().parent / 'data').exists() else Path.cwd() / 'data'
print('Libraries imported.')
print('Dataset path resolved.')

Libraries imported.
Dataset path resolved.


## 2. Load and inspect data

In [2]:
df = pd.read_csv(DATA_DIR / 'simple_linear_student_scores.csv')
df.head()

  record_id  study_hours  attendance_pct  exam_score
0      R001         1.09            72.4        49.4
1      R002         1.39            66.5        41.1
2      R003         1.10            67.2        41.9
3      R004         1.14            68.8        44.0
4      R005         0.91            71.8        43.9

In [3]:
pd.DataFrame({
    'column': df.columns,
    'dtype': [str(dtype) for dtype in df.dtypes],
    'missing_count': df.isna().sum().values,
    'unique_values': df.nunique().values,
})

           column    dtype  missing_count  unique_values
0       record_id   object              0             60
1     study_hours  float64              0             57
2  attendance_pct  float64              0             56
3      exam_score  float64              0             56

In [4]:
df[['study_hours', 'attendance_pct', 'exam_score']].describe().round(2)

       study_hours  attendance_pct  exam_score
count        60.00           60.00       60.00
mean          5.55           82.20       67.50
std           2.70            8.50       15.41
min           0.91           64.80       41.10
25%           3.19           76.40       54.92
50%           5.76           81.15       67.90
75%           8.27           89.88       82.40
max          10.02          100.00       97.20

## 3. Visualize the relationship

A scatter plot is the first sanity check for linearity.

In [5]:
plt.figure(figsize=(7,5))
plt.scatter(df['study_hours'], df['exam_score'], alpha=0.8)
plt.title('Study hours vs exam score')
plt.xlabel('Study hours')
plt.ylabel('Exam score')
plt.grid(True, alpha=0.3)
plt.show()
print('Rendered scatter plot: study_hours vs exam_score.')

Rendered scatter plot: study_hours vs exam_score.


## 4. Train-test split

In [6]:
X = df[['study_hours']]
y = df['exam_score']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=43)
print('X shape:', X.shape)
print('y shape:', y.shape)
print('Training rows:', len(X_train))
print('Testing rows :', len(X_test))

X shape: (60, 1)
y shape: (60,)
Training rows: 45
Testing rows : 15


## 5. Fit scikit-learn LinearRegression

In [7]:
model = LinearRegression()
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(f'Intercept: {model.intercept_:.4f}')
print(f'Slope    : {model.coef_[0]:.4f}')

Intercept: 39.0182
Slope    : 5.4579


In [8]:
x_line = np.linspace(df['study_hours'].min(), df['study_hours'].max(), 100)
x_line_df = pd.DataFrame({'study_hours': x_line})
y_line = model.predict(x_line_df)
plt.figure(figsize=(8,5))
plt.scatter(df['study_hours'], df['exam_score'], alpha=0.7, label='Actual data')
plt.plot(x_line, y_line, linewidth=2, label='Fitted line')
plt.title('Simple linear regression fitted line')
plt.xlabel('Study hours')
plt.ylabel('Exam score')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
print('Rendered fitted regression line.')

Rendered fitted regression line.


## 6. Model metrics

In [9]:
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)
metrics = pd.DataFrame({'metric': ['MAE', 'RMSE', 'R2'], 'value': [mae, rmse, r2]})
metrics

  metric     value
0    MAE  4.737797
1   RMSE  6.158041
2     R2  0.767041

## 7. Manual OLS formula

```text
slope = sum((x_i - mean_x)(y_i - mean_y)) / sum((x_i - mean_x)^2)
intercept = mean_y - slope * mean_x
```

In [10]:
x = X_train['study_hours'].to_numpy()
yt = y_train.to_numpy()
manual_slope = ((x - x.mean()) * (yt - yt.mean())).sum() / ((x - x.mean()) ** 2).sum()
manual_intercept = yt.mean() - manual_slope * x.mean()
pd.DataFrame({'implementation': ['manual_formula', 'sklearn'], 'intercept': [manual_intercept, model.intercept_], 'slope': [manual_slope, model.coef_[0]]})

   implementation  intercept     slope
0  manual_formula   39.01817  5.457874
1         sklearn   39.01817  5.457874

## 8. Prediction table and residuals

In [11]:
prediction_table = X_test.copy()
prediction_table['actual_exam_score'] = y_test.values
prediction_table['predicted_exam_score'] = pred
prediction_table['residual'] = prediction_table['actual_exam_score'] - prediction_table['predicted_exam_score']
prediction_table.sort_values('study_hours').round(2).head()

    study_hours  actual_exam_score  predicted_exam_score  residual
4          0.91               43.9                 43.98     -0.08
11         2.64               52.8                 53.43     -0.63
19         3.77               57.1                 59.59     -2.49
25         4.91               64.9                 65.82     -0.92
29         4.71               66.1                 64.73      1.37

In [12]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, pred, alpha=0.8)
min_value = min(y_test.min(), pred.min())
max_value = max(y_test.max(), pred.max())
plt.plot([min_value, max_value], [min_value, max_value], linewidth=2)
plt.title('Actual vs predicted exam score')
plt.xlabel('Actual exam score')
plt.ylabel('Predicted exam score')
plt.grid(True, alpha=0.3)
plt.show()
print('Rendered actual vs predicted plot.')

Rendered actual vs predicted plot.


In [13]:
residuals = y_test - pred
fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].scatter(pred, residuals, alpha=0.8)
axes[0].axhline(0, linewidth=1)
axes[0].set_title('Residuals vs predicted')
axes[1].hist(residuals, bins=10, edgecolor='black')
axes[1].set_title('Residual distribution')
plt.tight_layout()
plt.show()
print('Rendered residual plots.')

Rendered residual plots.


## 9. statsmodels OLS inference

In [14]:
ols = sm.OLS(y_train, sm.add_constant(X_train)).fit()
pd.DataFrame({'coefficient': ols.params, 'p_value': ols.pvalues, 'lower_95': ols.conf_int()[0], 'upper_95': ols.conf_int()[1]})

             coefficient       p_value   lower_95   upper_95
const          39.018170  5.496479e-29  36.167446  41.868895
study_hours     5.457874  3.132511e-26   4.991270   5.924477

## 10. Cross-validation

In [15]:
cv = KFold(n_splits=5, shuffle=True, random_state=43)
scores = cross_validate(LinearRegression(), X, y, cv=cv, scoring={'r2': 'r2', 'mae': 'neg_mean_absolute_error', 'rmse': 'neg_root_mean_squared_error'})
cv_results = pd.DataFrame({'fold': range(1, 6), 'r2': scores['test_r2'], 'mae': -scores['test_mae'], 'rmse': -scores['test_rmse']})
cv_results.round(4)

   fold      r2     mae    rmse
0     1  0.8675  3.3121  4.3664
1     2  0.9011  3.6148  4.4529
2     3  0.8479  4.0358  5.1797
3     4  0.9038  3.3282  4.2060
4     5  0.8639  3.7969  4.9008

## Final interpretation

The fitted slope is approximately **5.46**. In this synthetic dataset, one additional study hour is associated with an average increase of about **5.46 exam-score points**. Avoid causal wording unless the data comes from a causal experiment.

## Student exercises

1. Replace `study_hours` with `attendance_pct`.
2. Build a two-feature model using `study_hours` and `attendance_pct`.
3. Change the random seed and compare metrics.
4. Add a residual plot using `study_hours` on the x-axis.
5. Write a final conclusion with metrics and limitations.